In [1]:
import os
from dotenv import load_dotenv
import time
import requests
import numpy as np

import pandas as pd
import json
from math import ceil
load_dotenv()

True

In [2]:
# =============================
# GROQ API SETUP
# =============================
# export GROQ_API_KEY="your_key_here"
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"

if not GROQ_API_KEY:
    raise EnvironmentError("GROQ_API_KEY is not set")


In [3]:
# =============================
# CATEGORY DEFINITIONS (New Version)
# =============================
CATEGORY_DEFINITIONS = {
    "lokasi": "Keluar karena pindah lokasi geografis secara eksplisit (pindah kota, luar kota, luar negeri/relokasi).",

    "perbedaan_musim_kehidupan": "Keluar karena perubahan fase hidup jangka panjang (menikah, punya anak, jenjang studi baru, perubahan karier besar). Bukan konflik jadwal rutin.",

    "tidak_ada_respon": "Tidak atau minim respons saat dihubungi, Missing in Action (tidak balas, sulit dihubungi, tidak pernah hadir).",

    "tertanam_di_gereja_lain": "Memilih tetap tertanam atau aktif di gereja lain, bukan di JPCC.",

    "waktu_tidak_sesuai": "Benturan jadwal atau komitmen waktu rutin (jam kerja, shift, pulang malam, jadwal kuliah).",

    "alasan_DATE": "Masalah atau kondisi terkait kelompok DATE (tidak cocok, beda usia, pindah DATE, konflik leader/member, DATE close/bubar).",

    "Admin": "Kesalahan administratif atau perubahan data yang bukan keputusan pribadi anggota (human error, new comer, probation, false positive).",

    "others": "Alasan tidak jelas, wafat, terlalu singkat, ambigu, atau tidak termasuk kategori lain."
}   

In [ ]:
def classify_batch_with_groq_compound_mini(texts: list[str]) -> list[dict]:
    """
    Classify multiple texts in ONE request using Groq.
    Enforces output length by repairing or padding if needed.
    """

    prompt = f"""
You are a data annotation system.

Categories:
{json.dumps(CATEGORY_DEFINITIONS, ensure_ascii=False)}

Rules:
- Choose EXACTLY ONE category per text.
- Label MUST match one of the category keys.
- If relocation is explicitly mentioned → choose "lokasi".
- If routine schedule conflict → choose "waktu_tidak_sesuai".
- If long-term life phase change → choose "perbedaan_musim_kehidupan".
- If related to DATE group condition/conflict → choose "alasan_DATE".
- If unclear or insufficient information → choose "others".
- Do not infer beyond the text.

Return a JSON array (same order as input).
Each object must contain:
- final_label (string)
- confidence (0–100 integer)
- keywords (Max 3 short keywords)

Output ONLY valid JSON. No explanations.

Texts:
{json.dumps(texts, ensure_ascii=False)}
""" # prompt untuk model dengan instruksi yang jelas dan aturan yang ketat

    payload = {
        "model": "llama-3.3-70b-versatile",
        "messages": [
            {"role": "system", "content": "You are a precise classification engine."}, # instruksi sistem untuk menetapkan peran model
            {"role": "user", "content": prompt},
        ],
        "temperature": 0, # deterministik untuk klasifikasi konsisten
    }

    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json",
    }

    for attempt in range(5):
        response = requests.post(GROQ_URL, json=payload, headers=headers, timeout=60)

        if response.status_code == 429:
            wait_time = 5 * (attempt + 1)
            print(f"Rate limited. Sleeping {wait_time}s...")
            time.sleep(wait_time)
            continue

        response.raise_for_status()
        content = response.json()["choices"][0]["message"]["content"] # mengambil konten pesan dari respons JSON

        try:
            data = json.loads(content)
        except json.JSONDecodeError:
            raise ValueError("Model did not return valid JSON") # hardening: pastikan output bisa di-parse sebagai JSON

        # 🔒 HARDENING: length enforcement
        if not isinstance(data, list):
            raise ValueError("Model output is not a JSON array") # hardening: pastikan output adalah array JSON

        if len(data) > len(texts):
            data = data[:len(texts)] # hardening: potong jika output lebih panjang dari input

        if len(data) < len(texts):
            missing = len(texts) - len(data) # hardening: jika output lebih pendek, tambahkan placeholder untuk menjaga keselarasan
            data.extend([{
                "final_label": "",
                "confidence": "",
                "keywords": []
            }] * missing)

        return data # mengembalikan data yang sudah dipastikan sesuai dengan panjang input

    raise RuntimeError("Failed after retries")


In [6]:
# split data df menjadi 5 file variable yang berbeda
df = pd.read_csv("freetext.csv")
df_split = np.array_split(df, 5)

for i, split_df in enumerate(df_split):
    split_df.to_csv(f"freetext_split_{i+1}.csv", index=False) 


c:\Users\Jovan\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [6]:
df = pd.read_csv("freetext_split_1.csv")
df.shape

(476, 1)

In [ ]:
# =============================
# PIPELINE (BATCHED → STRUCTURED CSV)
# =============================
    
def run_pipeline(batch_size: int = 10):
    # df = pd.read_csv("freetext.csv").head(200)
    df = pd.read_csv("freetext_split_1.csv")
    results = []

    num_batches = ceil(len(df) / batch_size) # menghitung jumlah batch berdasarkan ukuran batch dan panjang dataframe

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size # menghitung indeks awal batch
        end = min(start + batch_size, len(df)) # menghitung indeks akhir batch dengan memastikan tidak melebihi panjang dataframe
        batch = df.iloc[start:end] # mengambil subset dataframe untuk batch saat ini

        batch_label_base = f"batch : {batch_idx + 1} of {num_batches}" # label untuk batch saat ini
        print(f"Processing {batch_label_base}")

        try:
            outputs = classify_batch_with_groq_compound_mini(
                batch["delete_reason"].tolist()
            )

            if len(outputs) != len(batch):
                raise ValueError("Output length mismatch") # hardening: pastikan output sesuai dengan batch size

        except Exception as e:
            print("Batch error:", e)
            outputs = [{} for _ in range(len(batch))] # fallback: kosongkan output untuk batch ini jika error

        for i, (_, row) in enumerate(batch.iterrows()):
            out = outputs[i]
            results.append({
                "delete_reason": row["delete_reason"],
                "batch": f"{batch_label_base} row: {i + 1}",
                "Final label": out.get("final_label", ""),
                "Confidence": f"{out.get('confidence', '')}%" if out else "",
                "Keywords": ", ".join(out.get("keywords", [])) if out else "",
            }) # menyimpan hasil dengan informasi batch dan row untuk traceability

    pd.DataFrame(results).to_csv("labeled_resultsLamma.csv", index=False) # menyimpan hasil akhir ke CSV
    print("Saved to labeled_resultsLamma.csv")


if __name__ == "__main__":
    run_pipeline(batch_size=10)


Processing batch : 1 of 48
Processing batch : 2 of 48
Processing batch : 3 of 48
Processing batch : 4 of 48
Processing batch : 5 of 48
Processing batch : 6 of 48
Processing batch : 7 of 48
Processing batch : 8 of 48
Processing batch : 9 of 48
Processing batch : 10 of 48
Processing batch : 11 of 48
Processing batch : 12 of 48
Processing batch : 13 of 48
Processing batch : 14 of 48
Processing batch : 15 of 48
Processing batch : 16 of 48
Rate limited. Sleeping 5s...
Processing batch : 17 of 48
Rate limited. Sleeping 5s...
Processing batch : 18 of 48
Processing batch : 19 of 48
Rate limited. Sleeping 5s...
Processing batch : 20 of 48
Rate limited. Sleeping 5s...
Processing batch : 21 of 48
Processing batch : 22 of 48
Rate limited. Sleeping 5s...
Processing batch : 23 of 48
Rate limited. Sleeping 5s...
Processing batch : 24 of 48
Processing batch : 25 of 48
Rate limited. Sleeping 5s...
Processing batch : 26 of 48
Rate limited. Sleeping 5s...
Processing batch : 27 of 48
Processing batch : 28

In [8]:
# =============================
# PIPELINE (BATCHED → STRUCTURED CSV)
# =============================

def run_pipeline(batch_size: int = 10):
    # df = pd.read_csv("freetext.csv").head(200)
    df = pd.read_csv("freetext_split_2.csv")
    results = []

    num_batches = ceil(len(df) / batch_size) # menghitung jumlah batch berdasarkan ukuran batch dan panjang dataframe

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size # menghitung indeks awal batch
        end = min(start + batch_size, len(df)) # menghitung indeks akhir batch dengan memastikan tidak melebihi panjang dataframe
        batch = df.iloc[start:end] # mengambil subset dataframe untuk batch saat ini

        batch_label_base = f"batch : {batch_idx + 1} of {num_batches}" # label untuk batch saat ini
        print(f"Processing {batch_label_base}")

        try:
            outputs = classify_batch_with_groq_compound_mini(
                batch["delete_reason"].tolist()
            )

            if len(outputs) != len(batch):
                raise ValueError("Output length mismatch") # hardening: pastikan output sesuai dengan batch size

        except Exception as e:
            print("Batch error:", e)
            outputs = [{} for _ in range(len(batch))] # fallback: kosongkan output untuk batch ini jika error

        for i, (_, row) in enumerate(batch.iterrows()):
            out = outputs[i]
            results.append({
                "delete_reason": row["delete_reason"],
                "batch": f"{batch_label_base} row: {i + 1}",
                "Final label": out.get("final_label", ""),
                "Confidence": f"{out.get('confidence', '')}%" if out else "",
                "Keywords": ", ".join(out.get("keywords", [])) if out else "",
            }) # menyimpan hasil dengan informasi batch dan row untuk traceability

    pd.DataFrame(results).to_csv("labeled_resultsLamma2.csv", index=False) # menyimpan hasil akhir ke CSV
    print("Saved to labeled_resultsLamma2.csv")


if __name__ == "__main__":
    run_pipeline(batch_size=10)


Processing batch : 1 of 48
Processing batch : 2 of 48
Processing batch : 3 of 48
Processing batch : 4 of 48
Processing batch : 5 of 48
Processing batch : 6 of 48
Processing batch : 7 of 48
Processing batch : 8 of 48
Processing batch : 9 of 48
Processing batch : 10 of 48
Processing batch : 11 of 48
Processing batch : 12 of 48
Processing batch : 13 of 48
Processing batch : 14 of 48
Processing batch : 15 of 48
Processing batch : 16 of 48
Rate limited. Sleeping 5s...
Processing batch : 17 of 48
Processing batch : 18 of 48
Rate limited. Sleeping 5s...
Processing batch : 19 of 48
Rate limited. Sleeping 5s...
Processing batch : 20 of 48
Rate limited. Sleeping 5s...
Processing batch : 21 of 48
Processing batch : 22 of 48
Rate limited. Sleeping 5s...
Processing batch : 23 of 48
Rate limited. Sleeping 5s...
Processing batch : 24 of 48
Processing batch : 25 of 48
Rate limited. Sleeping 5s...
Processing batch : 26 of 48
Rate limited. Sleeping 5s...
Processing batch : 27 of 48
Processing batch : 28

In [7]:
# =============================
# PIPELINE (BATCHED → STRUCTURED CSV)
# =============================

def run_pipeline(batch_size: int = 10):
    # df = pd.read_csv("freetext.csv").head(200)
    df = pd.read_csv("freetext_split_3.csv")
    results = []

    num_batches = ceil(len(df) / batch_size) # menghitung jumlah batch berdasarkan ukuran batch dan panjang dataframe

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size # menghitung indeks awal batch
        end = min(start + batch_size, len(df)) # menghitung indeks akhir batch dengan memastikan tidak melebihi panjang dataframe
        batch = df.iloc[start:end] # mengambil subset dataframe untuk batch saat ini

        batch_label_base = f"batch : {batch_idx + 1} of {num_batches}" # label untuk batch saat ini
        print(f"Processing {batch_label_base}")

        try:
            outputs = classify_batch_with_groq_compound_mini(
                batch["delete_reason"].tolist()
            )

            if len(outputs) != len(batch):
                raise ValueError("Output length mismatch") # hardening: pastikan output sesuai dengan batch size

        except Exception as e:
            print("Batch error:", e)
            outputs = [{} for _ in range(len(batch))] # fallback: kosongkan output untuk batch ini jika error

        for i, (_, row) in enumerate(batch.iterrows()):
            out = outputs[i]
            results.append({
                "delete_reason": row["delete_reason"],
                "batch": f"{batch_label_base} row: {i + 1}",
                "Final label": out.get("final_label", ""),
                "Confidence": f"{out.get('confidence', '')}%" if out else "",
                "Keywords": ", ".join(out.get("keywords", [])) if out else "",
            }) # menyimpan hasil dengan informasi batch dan row untuk traceability

    pd.DataFrame(results).to_csv("labeled_resultsLamma3.csv", index=False) # menyimpan hasil akhir ke CSV
    print("Saved to labeled_resultsLamma3.csv")


if __name__ == "__main__":
    run_pipeline(batch_size=10)


Processing batch : 1 of 48
Processing batch : 2 of 48
Processing batch : 3 of 48
Processing batch : 4 of 48
Processing batch : 5 of 48
Processing batch : 6 of 48
Processing batch : 7 of 48
Processing batch : 8 of 48
Processing batch : 9 of 48
Processing batch : 10 of 48
Processing batch : 11 of 48
Processing batch : 12 of 48
Processing batch : 13 of 48
Processing batch : 14 of 48
Rate limited. Sleeping 5s...
Processing batch : 15 of 48
Rate limited. Sleeping 5s...
Processing batch : 16 of 48
Processing batch : 17 of 48
Rate limited. Sleeping 5s...
Processing batch : 18 of 48
Rate limited. Sleeping 5s...
Processing batch : 19 of 48
Rate limited. Sleeping 5s...
Processing batch : 20 of 48
Rate limited. Sleeping 5s...
Processing batch : 21 of 48
Rate limited. Sleeping 5s...
Processing batch : 22 of 48
Processing batch : 23 of 48
Rate limited. Sleeping 5s...
Processing batch : 24 of 48
Rate limited. Sleeping 5s...
Processing batch : 25 of 48
Rate limited. Sleeping 5s...
Processing batch : 

In [7]:
# =============================
# PIPELINE (BATCHED → STRUCTURED CSV)
# =============================

def run_pipeline(batch_size: int = 10):
    # df = pd.read_csv("freetext.csv").head(200)
    df = pd.read_csv("freetext_split_4.csv")
    results = []

    num_batches = ceil(len(df) / batch_size) # menghitung jumlah batch berdasarkan ukuran batch dan panjang dataframe

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size # menghitung indeks awal batch
        end = min(start + batch_size, len(df)) # menghitung indeks akhir batch dengan memastikan tidak melebihi panjang dataframe
        batch = df.iloc[start:end] # mengambil subset dataframe untuk batch saat ini

        batch_label_base = f"batch : {batch_idx + 1} of {num_batches}" # label untuk batch saat ini
        print(f"Processing {batch_label_base}")

        try:
            outputs = classify_batch_with_groq_compound_mini(
                batch["delete_reason"].tolist()
            )

            if len(outputs) != len(batch):
                raise ValueError("Output length mismatch") # hardening: pastikan output sesuai dengan batch size

        except Exception as e:
            print("Batch error:", e)
            outputs = [{} for _ in range(len(batch))] # fallback: kosongkan output untuk batch ini jika error

        for i, (_, row) in enumerate(batch.iterrows()):
            out = outputs[i]
            results.append({
                "delete_reason": row["delete_reason"],
                "batch": f"{batch_label_base} row: {i + 1}",
                "Final label": out.get("final_label", ""),
                "Confidence": f"{out.get('confidence', '')}%" if out else "",
                "Keywords": ", ".join(out.get("keywords", [])) if out else "",
            }) # menyimpan hasil dengan informasi batch dan row untuk traceability

    pd.DataFrame(results).to_csv("labeled_resultsLamma4.csv", index=False) # menyimpan hasil akhir ke CSV
    print("Saved to labeled_resultsLamma4.csv")


if __name__ == "__main__":
    run_pipeline(batch_size=10)


Processing batch : 1 of 48
Processing batch : 2 of 48
Processing batch : 3 of 48
Processing batch : 4 of 48
Processing batch : 5 of 48
Processing batch : 6 of 48
Processing batch : 7 of 48
Processing batch : 8 of 48
Processing batch : 9 of 48
Processing batch : 10 of 48
Processing batch : 11 of 48
Processing batch : 12 of 48
Processing batch : 13 of 48
Processing batch : 14 of 48
Rate limited. Sleeping 5s...
Processing batch : 15 of 48
Rate limited. Sleeping 5s...
Processing batch : 16 of 48
Rate limited. Sleeping 5s...
Processing batch : 17 of 48
Rate limited. Sleeping 5s...
Processing batch : 18 of 48
Rate limited. Sleeping 5s...
Processing batch : 19 of 48
Rate limited. Sleeping 5s...
Processing batch : 20 of 48
Processing batch : 21 of 48
Rate limited. Sleeping 5s...
Processing batch : 22 of 48
Rate limited. Sleeping 5s...
Processing batch : 23 of 48
Rate limited. Sleeping 5s...
Processing batch : 24 of 48
Rate limited. Sleeping 5s...
Processing batch : 25 of 48
Rate limited. Sleep

In [8]:
# =============================
# PIPELINE (BATCHED → STRUCTURED CSV)
# =============================

def run_pipeline(batch_size: int = 10):
    # df = pd.read_csv("freetext.csv").head(200)
    df = pd.read_csv("freetext_split_5.csv")
    results = []

    num_batches = ceil(len(df) / batch_size) # menghitung jumlah batch berdasarkan ukuran batch dan panjang dataframe

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size # menghitung indeks awal batch
        end = min(start + batch_size, len(df)) # menghitung indeks akhir batch dengan memastikan tidak melebihi panjang dataframe
        batch = df.iloc[start:end] # mengambil subset dataframe untuk batch saat ini

        batch_label_base = f"batch : {batch_idx + 1} of {num_batches}" # label untuk batch saat ini
        print(f"Processing {batch_label_base}")

        try:
            outputs = classify_batch_with_groq_compound_mini(
                batch["delete_reason"].tolist()
            )

            if len(outputs) != len(batch):
                raise ValueError("Output length mismatch") # hardening: pastikan output sesuai dengan batch size

        except Exception as e:
            print("Batch error:", e)
            outputs = [{} for _ in range(len(batch))] # fallback: kosongkan output untuk batch ini jika error

        for i, (_, row) in enumerate(batch.iterrows()):
            out = outputs[i]
            results.append({
                "delete_reason": row["delete_reason"],
                "batch": f"{batch_label_base} row: {i + 1}",
                "Final label": out.get("final_label", ""),
                "Confidence": f"{out.get('confidence', '')}%" if out else "",
                "Keywords": ", ".join(out.get("keywords", [])) if out else "",
            }) # menyimpan hasil dengan informasi batch dan row untuk traceability

    pd.DataFrame(results).to_csv("labeled_resultsLamma5.csv", index=False) # menyimpan hasil akhir ke CSV
    print("Saved to labeled_resultsLamma5.csv")


if __name__ == "__main__":
    run_pipeline(batch_size=10)


Processing batch : 1 of 48
Processing batch : 2 of 48
Processing batch : 3 of 48
Processing batch : 4 of 48
Processing batch : 5 of 48
Processing batch : 6 of 48
Processing batch : 7 of 48
Processing batch : 8 of 48
Processing batch : 9 of 48
Processing batch : 10 of 48
Processing batch : 11 of 48
Processing batch : 12 of 48
Processing batch : 13 of 48
Processing batch : 14 of 48
Rate limited. Sleeping 5s...
Processing batch : 15 of 48
Rate limited. Sleeping 5s...
Processing batch : 16 of 48
Processing batch : 17 of 48
Rate limited. Sleeping 5s...
Processing batch : 18 of 48
Rate limited. Sleeping 5s...
Processing batch : 19 of 48
Rate limited. Sleeping 5s...
Processing batch : 20 of 48
Rate limited. Sleeping 5s...
Processing batch : 21 of 48
Rate limited. Sleeping 5s...
Processing batch : 22 of 48
Rate limited. Sleeping 5s...
Processing batch : 23 of 48
Processing batch : 24 of 48
Rate limited. Sleeping 5s...
Processing batch : 25 of 48
Rate limited. Sleeping 5s...
Processing batch : 

In [8]:
# =============================
# PIPELINE (BATCHED → STRUCTURED CSV)
# =============================
    
def run_pipeline(batch_size: int = 10):
    # df = pd.read_csv("freetext.csv").head(200)
    df = pd.read_csv("labeled_resultsLammaFull.csv")
    df=df.head(10)
    results = []

    num_batches = ceil(len(df) / batch_size) # menghitung jumlah batch berdasarkan ukuran batch dan panjang dataframe

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size # menghitung indeks awal batch
        end = min(start + batch_size, len(df)) # menghitung indeks akhir batch dengan memastikan tidak melebihi panjang dataframe
        batch = df.iloc[start:end] # mengambil subset dataframe untuk batch saat ini

        batch_label_base = f"batch : {batch_idx + 1} of {num_batches}" # label untuk batch saat ini
        print(f"Processing {batch_label_base}")

        try:
            outputs = classify_batch_with_groq_compound_mini(
                batch["delete_reason"].tolist()
            )

            if len(outputs) != len(batch):
                raise ValueError("Output length mismatch") # hardening: pastikan output sesuai dengan batch size

        except Exception as e:
            print("Batch error:", e)
            outputs = [{} for _ in range(len(batch))] # fallback: kosongkan output untuk batch ini jika error

        for i, (_, row) in enumerate(batch.iterrows()):
            out = outputs[i]
            results.append({
                "delete_reason": row["delete_reason"],
                "batch": f"{batch_label_base} row: {i + 1}",
                "Final label": out.get("final_label", ""),
                "Confidence": f"{out.get('confidence', '')}%" if out else "",
                "Keywords": ", ".join(out.get("keywords", [])) if out else "",
            }) # menyimpan hasil dengan informasi batch dan row untuk traceability

    pd.DataFrame(results).to_csv("labeled_resultsLammaDone.csv", index=False) # menyimpan hasil akhir ke CSV
    print("Saved to labeled_resultsLammaDone.csv")


if __name__ == "__main__":
    run_pipeline(batch_size=10)


Processing batch : 1 of 1
Saved to labeled_resultsLammaDone.csv


In [8]:
# =============================
# PIPELINE (BATCHED → STRUCTURED CSV)
# =============================
    
def run_pipeline(batch_size: int = 10):
    # df = pd.read_csv("freetext.csv").head(200)
    df = pd.read_csv("consistencydata.csv")
    results = []

    num_batches = ceil(len(df) / batch_size) # menghitung jumlah batch berdasarkan ukuran batch dan panjang dataframe

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size # menghitung indeks awal batch
        end = min(start + batch_size, len(df)) # menghitung indeks akhir batch dengan memastikan tidak melebihi panjang dataframe
        batch = df.iloc[start:end] # mengambil subset dataframe untuk batch saat ini

        batch_label_base = f"batch : {batch_idx + 1} of {num_batches}" # label untuk batch saat ini
        print(f"Processing {batch_label_base}")

        try:
            outputs = classify_batch_with_groq_compound_mini(
                batch["delete_reason"].tolist()
            )

            if len(outputs) != len(batch):
                raise ValueError("Output length mismatch") # hardening: pastikan output sesuai dengan batch size

        except Exception as e:
            print("Batch error:", e)
            outputs = [{} for _ in range(len(batch))] # fallback: kosongkan output untuk batch ini jika error

        for i, (_, row) in enumerate(batch.iterrows()):
            out = outputs[i]
            results.append({
                "delete_reason": row["delete_reason"],
                "batch": f"{batch_label_base} row: {i + 1}",
                "Final label": out.get("final_label", ""),
                "Confidence": f"{out.get('confidence', '')}%" if out else "",
                "Keywords": ", ".join(out.get("keywords", [])) if out else "",
            }) # menyimpan hasil dengan informasi batch dan row untuk traceability

    pd.DataFrame(results).to_csv("ConsistencyLamma.csv", index=False) # menyimpan hasil akhir ke CSV
    print("Saved to ConsistencyLamma.csv")


if __name__ == "__main__":
    run_pipeline(batch_size=10)


Processing batch : 1 of 25
Processing batch : 2 of 25
Processing batch : 3 of 25
Processing batch : 4 of 25
Processing batch : 5 of 25
Processing batch : 6 of 25
Processing batch : 7 of 25
Processing batch : 8 of 25
Processing batch : 9 of 25
Processing batch : 10 of 25
Processing batch : 11 of 25
Processing batch : 12 of 25
Processing batch : 13 of 25
Processing batch : 14 of 25
Processing batch : 15 of 25
Processing batch : 16 of 25
Rate limited. Sleeping 5s...
Processing batch : 17 of 25
Rate limited. Sleeping 5s...
Processing batch : 18 of 25
Rate limited. Sleeping 5s...
Processing batch : 19 of 25
Processing batch : 20 of 25
Rate limited. Sleeping 5s...
Processing batch : 21 of 25
Rate limited. Sleeping 5s...
Processing batch : 22 of 25
Processing batch : 23 of 25
Rate limited. Sleeping 5s...
Processing batch : 24 of 25
Rate limited. Sleeping 5s...
Processing batch : 25 of 25
Saved to ConsistencyLamma.csv
